In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import re

def parse_bid_value(bid_value):
    try:
        val = bid_value.lower().replace('₹','').strip()
        if 'cr' in val:
            return float(val.replace('cr', '').strip()) * 1e7
        elif 'lakh' in val:
            return float(val.replace('lakh', '').strip()) * 1e5
        elif 'thousand' in val:
            return float(val.replace('thousand', '').strip()) * 1e3
        elif 'million' in val:
            return float(val.replace('million', '').strip()) * 1e6
        elif 'billion' in val:
            return float(val.replace('billion', '').strip()) * 1e9
        else:
            return float(val.replace(',',''))  # raw number
    except:
        return None

# Selenium setup
options = webdriver.ChromeOptions()
# Uncomment to run headless
# options.add_argument('--headless')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Target URL
url = "https://www.tender247.com/tenders?state=himachal+pradesh&advancesearch=true"
driver.get(url)
time.sleep(4)

# Scroll to load more tenders
for _ in range(25):  # Adjust as needed
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)

cards = driver.find_elements(By.CSS_SELECTOR, 'div.w-\\[100\\%\\].float-left')
print(f"Found {len(cards)} cards, parsing...")

data = []

for card in cards:
    card_html = card.get_attribute('innerHTML')
    if "Bid Value:" not in card_html or "T247 ID-" not in card_html:
        continue

    try:
        full_text = card.get_attribute("innerText")

        # Bid Value (robust using regex)
        try:
            bid_value_match = re.search(r"Bid Value:\s*₹?\s*[\d.,]+\s*(Cr|Lakh|Thousand|Million|Billion)?", full_text, re.IGNORECASE)
            bid_value = bid_value_match.group(0).replace("Bid Value:", "").strip() if bid_value_match else ""
        except:
            bid_value = ""

        # EMD
        try:
            emd = card.find_element(By.XPATH, ".//span[contains(text(),'EMD:')]/following-sibling::div").text.strip()
        except:
            emd = ""

        # Closing Date & Days Left
        try:
            closing_date = ""
            days_left = ""
            float_divs = card.find_elements(By.XPATH, ".//div[contains(@class, 'float-left') and contains(@class, 'mr-2')]")
            for fd in float_divs:
                m = re.match(r"\s*([0-9]{2}-[0-9]{2}-[0-9]{4})", fd.text)
                if m:
                    closing_date = m.group(1)
                    try:
                        days_left_span = fd.find_element(By.TAG_NAME, "span")
                        days_left = days_left_span.text.strip()
                    except:
                        days_left = ""
                    break
        except:
            closing_date = ""
            days_left = ""

        # T247 ID
        try:
            t247_id = card.find_element(By.XPATH, ".//span[contains(text(),'T247 ID-')]/..").text.replace("T247 ID-", "").replace("|", "").strip()
        except:
            t247_id = ""

        # Description
        try:
            description = card.find_element(By.XPATH, ".//p/a/span").text.strip()
        except:
            description = ""

        # Location
        try:
            all_h3s = card.find_elements(By.TAG_NAME, "h3")
            location = ""
            for h3 in all_h3s:
                svgs = h3.find_elements(By.TAG_NAME, "svg")
                if svgs and "India" in h3.text:
                    inner_html = h3.get_attribute('innerHTML')
                    if "</svg>" in inner_html:
                        location = inner_html.split("</svg>")[-1].strip()
                    else:
                        location = h3.text.strip()
                    break
        except:
            location = ""

        # Bid Link
        try:
            bid_link = card.find_element(By.XPATH, ".//a[contains(@href, '/tender-details/')]").get_attribute("href")
            bid_link = "https://www.tender247.com" + bid_link if bid_link.startswith("/") else bid_link
        except:
            bid_link = ""

        data.append({
            "T247 ID": t247_id,
            "Bid Value": bid_value,
            "EMD": emd,
            "Closing Date": closing_date,
            "Days Left": days_left,
            "Description": description,
            "Location": location,
            "Bid Link": bid_link,
        })

    except Exception as e:
        print("Error on card:", e)
        continue

driver.quit()

# Create DataFrame
df = pd.DataFrame(data)
df["Bid Value (INR)"] = df["Bid Value"].apply(parse_bid_value)

# Save to Excel
df.to_excel("Tender247_himachal+pradesh1.xlsx", index=False)
print("✅ Scraping done! File saved: Tender247_himachal+pradesh1.xlsx")


Found 1042 cards, parsing...
✅ Scraping done! File saved: Tender247_himachal+pradesh1.xlsx


In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import re

def parse_bid_value(bid_value):
    try:
        val = bid_value.lower().replace('₹','').strip()
        if 'cr' in val:
            return float(val.replace('cr', '').strip()) * 1e7
        elif 'lakh' in val:
            return float(val.replace('lakh', '').strip()) * 1e5
        elif 'thousand' in val:
            return float(val.replace('thousand', '').strip()) * 1e3
        elif 'million' in val:
            return float(val.replace('million', '').strip()) * 1e6
        elif 'billion' in val:
            return float(val.replace('billion', '').strip()) * 1e9
        else:
            return float(val.replace(',',''))  # raw number
    except:
        return None

# Setup Selenium
options = webdriver.ChromeOptions()
# Uncomment to run headless
# options.add_argument('--headless')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Load URL
url = "https://www.tender247.com/tenders?state=gujarat&advancesearch=true"
driver.get(url)
time.sleep(4)

# Scroll to load more tenders
for _ in range(25):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)

# Extract all cards
cards = driver.find_elements(By.CSS_SELECTOR, 'div.w-\\[100\\%\\].float-left')
print(f"Found {len(cards)} cards, parsing...")

data = []

for card in cards:
    try:
        full_text = card.get_attribute("innerText")

        # Bid Value
        try:
            bid_value_match = re.search(r"Bid Value:\s*₹?\s*[\d.,]+\s*(Cr|Lakh|Thousand|Million|Billion)?", full_text, re.IGNORECASE)
            bid_value = bid_value_match.group(0).replace("Bid Value:", "").strip() if bid_value_match else ""
        except:
            bid_value = ""

        # EMD
        try:
            emd = card.find_element(By.XPATH, ".//span[contains(text(),'EMD:')]/following-sibling::div").text.strip()
        except:
            emd = ""

        # Closing Date & Days Left
        try:
            closing_date = ""
            days_left = ""
            float_divs = card.find_elements(By.XPATH, ".//div[contains(@class, 'float-left') and contains(@class, 'mr-2')]")
            for fd in float_divs:
                m = re.match(r"\s*([0-9]{2}-[0-9]{2}-[0-9]{4})", fd.text)
                if m:
                    closing_date = m.group(1)
                    try:
                        days_left_span = fd.find_element(By.TAG_NAME, "span")
                        days_left = days_left_span.text.strip()
                    except:
                        days_left = ""
                    break
        except:
            closing_date = ""
            days_left = ""

        # T247 ID
        try:
            t247_id = card.find_element(By.XPATH, ".//span[contains(text(),'T247 ID-')]/..").text.replace("T247 ID-", "").replace("|", "").strip()
        except:
            t247_id = ""

        # Description
        try:
            description = card.find_element(By.XPATH, ".//p/a/span").text.strip()
        except:
            description = ""

        # Location
        try:
            all_h3s = card.find_elements(By.TAG_NAME, "h3")
            location = ""
            for h3 in all_h3s:
                svgs = h3.find_elements(By.TAG_NAME, "svg")
                if svgs and "India" in h3.text:
                    inner_html = h3.get_attribute('innerHTML')
                    if "</svg>" in inner_html:
                        location = inner_html.split("</svg>")[-1].strip()
                    else:
                        location = h3.text.strip()
                    break
        except:
            location = ""

        # Bid Link
        try:
            bid_link = card.find_element(By.XPATH, ".//a[contains(@href, '/tender-details/')]").get_attribute("href")
            bid_link = "https://www.tender247.com" + bid_link if bid_link.startswith("/") else bid_link
        except:
            bid_link = ""

        data.append({
            "T247 ID": t247_id,
            "Bid Value": bid_value,
            "EMD": emd,
            "Closing Date": closing_date,
            "Days Left": days_left,
            "Description": description,
            "Location": location,
            "Bid Link": bid_link,
        })

    except Exception as e:
        print("Error on card:", e)
        continue

driver.quit()

# Create DataFrame and parse INR value
df = pd.DataFrame(data)
df["Bid Value (INR)"] = df["Bid Value"].apply(parse_bid_value)

# Save to Excel
filename = "Tender247_gujarat1.xlsx"
df.to_excel(filename, index=False)

# Summary
print(f"\n✅ Scraping done! File saved: {filename}")
print(f"Total cards parsed: {len(cards)}")
print(f"Total rows saved: {len(df)}")
print(f"Missing Bid Value: {df['Bid Value'].isna().sum() + (df['Bid Value'] == '').sum()}")
print(f"Missing T247 ID: {df['T247 ID'].isna().sum() + (df['T247 ID'] == '').sum()}")


Found 1082 cards, parsing...

✅ Scraping done! File saved: Tender247_gujarat1.xlsx
Total cards parsed: 1082
Total rows saved: 1082
Missing Bid Value: 542
Missing T247 ID: 542


In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

url = "https://www.tender247.com/tenders?state=delhi&advancesearch=true"
driver.get(url)
time.sleep(4)

# Scroll to load more tenders
for _ in range(25):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

# Use a broad selector (adjust as needed if cards not found)
cards = driver.find_elements(By.CSS_SELECTOR, 'div.float-left')

print(f"Found {len(cards)} cards, extracting all card text...")

data = []
for card in cards:
    card_text = card.text.strip()
    if card_text:
        data.append({'Card Text': card_text})

driver.quit()

df = pd.DataFrame(data)
df.to_excel("Tender247_delhi_ALL.xlsx", index=False)
print("All card text saved to Tender247_delhi_ALL.xlsx")


Found 4036 cards, extracting all card text...
All card text saved to Tender247_delhi_ALL.xlsx
